In [21]:
import pathlib as pl

# adapt relative path until it loads
cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

PLOT_ROOT = CONFIG["plot_root"].joinpath(NB_PATH.stem)
TABLE_ROOT = CONFIG["table_root"].joinpath(NB_PATH.stem)
NB_CACHE_FOLDER = CONFIG["nb_cache_folder"]
DATA_ROOT = CONFIG["project_data"]

import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# NB-specific code

import pandas as pd

refs = {
    "t2tv2": "T2Tv2",
    "hg38": "GRCh38"
}

LABEL_FILE_SOURCE = DATA_ROOT.joinpath("wf-data/region-labels/final_annot/2026-02_final-rev")

STRUCT_ERRORS = [["ERRSTRUCT"], ["ERRSTRUCT", "NGAP"]]

def assign_seq_type(seqname):
    if seqname.endswith("_chrY"):
        st = "main"
    else:
        st = "rand"
    return st

sample_stats = []
for ref in refs.keys():
    if ref != "t2tv2":
        continue
    label_files = LABEL_FILE_SOURCE.joinpath(ref).glob("*.bed")
    for label_file in label_files:
        sample = label_file.name.split(".")[0]
        df = pd.read_csv(label_file, sep="\t", header=0)
        df.rename({"#seq": "seq"}, axis=1, inplace=True)
        df["seqtype"] = df["seq"].apply(assign_seq_type)
        df["length"] = df["end"] - df["start"]
        stats = {"sample": sample}
        for st in [["main"], ["main", "rand"]]:
            st_label = "_".join(st)
            sub = df.loc[df["seqtype"].isin(st), :].copy()
            total_length = sub.groupby("seq")["end"].max().sum()
            assert total_length > 0
            stats[f"{st_label}_total_bp"] = total_length
            for error_labels in STRUCT_ERRORS:
                err_label = "_".join(error_labels)
                select_err = sub["name"].isin(error_labels)
                total_err = sub.loc[select_err, "length"].sum()
                stats[f"{st_label}_{err_label}_total_bp"] = total_err
                err_rate = round(total_err / total_length * int(1e6), 6)
                stats[f"{st_label}_{err_label}_per_Mbp"] = err_rate
        sample_stats.append(stats)

sample_stats = pd.DataFrame.from_records(sample_stats)
sample_stats.sort_values(["sample"], inplace=True)

relevant_stats = [
    "main_rand_ERRSTRUCT_NGAP_per_Mbp",
    "main_rand_ERRSTRUCT_per_Mbp"
]

for rs in relevant_stats:
    print(rs)
    print(sample_stats[rs].median().round(1))

out_tsv = TABLE_ROOT.joinpath("tab-error_rate", "sample_struct_error_rate.tsv")
out_tsv.parent.mkdir(exist_ok=True, parents=True)

sample_stats.to_csv(out_tsv, sep="\t", header=True, index=False)
            
            

main_rand_ERRSTRUCT_NGAP_per_Mbp
1175.0
main_rand_ERRSTRUCT_per_Mbp
14.9
